# Track A 10: System Environment and Reproducibility Specification Audit

**Project:** Holistic Optimization of SLMs for Energy-Efficient Edge Inference  
**Purpose:** Extract, audit, and document all hardware specifications, operating system parameters, GPU telemetry settings, software library versions, model checkpoint hashes, and evaluation protocol constants required for complete scientific reproducibility.

### Output Artifacts
Running this notebook generates three standalone reproducibility files:
1. `reproducibility_specs_full.json` — Hierarchical machine-readable specification manifest.
2. `reproducibility_table.tex` — Publication-ready LaTeX table formatted for the paper.
3. `reproducibility_summary.md` — Markdown summary reference sheet.

---

## 1. Setup and Diagnostic Imports

In [ ]:
import os
import sys
import json
import platform
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

# Check and safely inspect scientific stack
def safe_import(name):
    try:
        mod = __import__(name)
        return mod, getattr(mod, '__version__', 'installed')
    except ImportError:
        return None, 'Not Installed'

torch_mod, torch_ver = safe_import('torch')
np_mod, np_ver = safe_import('numpy')
sp_mod, sp_ver = safe_import('scipy')
pd_mod, pd_ver = safe_import('pandas')

print(f"Python Version: {sys.version}")
print(f"PyTorch: {torch_ver}")
print(f"NumPy: {np_ver}")
print(f"SciPy: {sp_ver}")
print(f"Pandas: {pd_ver}")

cuda_available = torch_mod.cuda.is_available() if torch_mod else False
print(f"CUDA Available in PyTorch: {cuda_available}")
if cuda_available:
    print(f"CUDA Device: {torch_mod.cuda.get_device_name(0)}")
    print(f"CUDA Runtime Version: {torch_mod.version.cuda}")


## 2. Host Operating System and Compute Specs

In [ ]:
def get_cpu_info():
    cpu_info = {
        "processor": platform.processor() or "Unknown",
        "architecture": platform.machine(),
        "logical_cores": os.cpu_count(),
    }
    # Try reading /proc/cpuinfo on Linux
    cpuinfo_path = Path("/proc/cpuinfo")
    if cpuinfo_path.exists():
        for line in cpuinfo_path.read_text().splitlines():
            if "model name" in line:
                cpu_info["model_name"] = line.split(":", 1)[1].strip()
                break
    return cpu_info

def get_ram_info():
    ram_info = {}
    meminfo_path = Path("/proc/meminfo")
    if meminfo_path.exists():
        for line in meminfo_path.read_text().splitlines():
            if "MemTotal:" in line:
                total_kb = int(line.split()[1])
                ram_info["total_gb"] = round(total_kb / (1024 * 1024), 2)
            elif "MemAvailable:" in line:
                avail_kb = int(line.split()[1])
                ram_info["available_gb"] = round(avail_kb / (1024 * 1024), 2)
    else:
        try:
            import psutil
            v = psutil.virtual_memory()
            ram_info["total_gb"] = round(v.total / (1024**3), 2)
            ram_info["available_gb"] = round(v.available / (1024**3), 2)
        except ImportError:
            ram_info["total_gb"] = "N/A"
    return ram_info

host_specs = {
    "os_system": platform.system(),
    "os_release": platform.release(),
    "os_version": platform.version(),
    "platform": platform.platform(),
    "cpu": get_cpu_info(),
    "ram": get_ram_info(),
}

print(json.dumps(host_specs, indent=2))


## 3. NVIDIA GPU and NVML Hardware Diagnostics

In [ ]:
gpu_specs = {}

try:
    if not (torch_mod and torch_mod.cuda.is_available()):
        raise RuntimeError("CUDA is not available in the current PyTorch environment; falling back to pinned registry.")
    import pynvml
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    
    device_name = pynvml.nvmlDeviceGetName(handle)
    if isinstance(device_name, bytes):
        device_name = device_name.decode("utf-8")
        
    driver_ver = pynvml.nvmlSystemGetDriverVersion()
    if isinstance(driver_ver, bytes):
        driver_ver = driver_ver.decode("utf-8")
        
    nvml_ver = pynvml.nvmlSystemGetNVMLVersion()
    if isinstance(nvml_ver, bytes):
        nvml_ver = nvml_ver.decode("utf-8")
        
    mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    
    # Power limits
    try:
        power_limit_mw = pynvml.nvmlDeviceGetPowerManagementLimit(handle)
        power_limit_w = power_limit_mw / 1000.0
    except Exception:
        power_limit_w = 70.0
        
    try:
        default_power_limit_mw = pynvml.nvmlDeviceGetPowerManagementDefaultLimit(handle)
        default_power_limit_w = default_power_limit_mw / 1000.0
    except Exception:
        default_power_limit_w = 70.0
        
    # Clocks and thermal
    try:
        sm_clock = pynvml.nvmlDeviceGetClockInfo(handle, pynvml.NVML_CLOCK_SM)
        mem_clock = pynvml.nvmlDeviceGetClockInfo(handle, pynvml.NVML_CLOCK_MEM)
    except Exception:
        sm_clock, mem_clock = "N/A", "N/A"
        
    try:
        temp_c = pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU)
    except Exception:
        temp_c = "N/A"
        
    # Compute capability from PyTorch
    major, minor = torch_mod.cuda.get_device_capability(0)
    
    gpu_specs = {
        "device_name": device_name,
        "driver_version": driver_ver,
        "nvml_version": nvml_ver,
        "compute_capability": f"{major}.{minor}",
        "vram_total_bytes": mem_info.total,
        "vram_total_gib": round(mem_info.total / (1024**3), 2),
        "nominal_vram_gb": 16,
        "usable_vram_gib": round(mem_info.total / (1024**3), 2),
        "enforced_power_limit_w": power_limit_w,
        "default_power_limit_w": default_power_limit_w,
        "sm_clock_mhz": sm_clock,
        "mem_clock_mhz": mem_clock,
        "current_temp_c": temp_c,
        "power_polling_interval_s": 0.1,
        "power_polling_frequency_hz": 10.0,
        "power_capping_regimes_w": ["raw (unclipped)", 82.0, 70.0],
        "power_integration_method": "Trapezoidal Riemann sum (integral P(t) dt)",
    }
    pynvml.nvmlShutdown()
except Exception as e:
    print(f"NVML / GPU direct inspection note: {e}")
    print("Loading canonical pinned GPU specs from experiment registry...")
    gpu_specs = {
        "device_name": "NVIDIA Tesla T4",
        "driver_version": "550.54.15",
        "cuda_version": "12.8",
        "compute_capability": "7.5 (Turing Architecture)",
        "sm_count": 40,
        "nominal_vram_gb": 16,
        "usable_vram_gib": 14.56,
        "memory_bus_width_bits": 256,
        "nominal_bandwidth_gb_s": 320.0,
        "enforced_power_limit_w": 70.0,
        "default_power_limit_w": 70.0,
        "max_observed_power_transient_llama_w": 124.496,
        "max_observed_power_transient_qwen_w": 116.109,
        "power_polling_interval_s": 0.1,
        "power_polling_frequency_hz": 10.0,
        "power_capping_regimes_w": ["raw (unclipped)", 82.0, 70.0],
        "power_integration_method": "Trapezoidal Riemann sum (integral P(t) dt)",
    }

print(json.dumps(gpu_specs, indent=2))


## 4. Software Stack and Package Dependency Audit

In [ ]:
def get_pkg_version(module_name):
    try:
        mod = __import__(module_name)
        return getattr(mod, "__version__", "installed")
    except ImportError:
        return "Not Installed"

software_stack = {
    "python": platform.python_version(),
    "torch": get_pkg_version("torch"),
    "torchvision": get_pkg_version("torchvision"),
    "transformers": get_pkg_version("transformers"),
    "accelerate": get_pkg_version("accelerate"),
    "gptqmodel": get_pkg_version("gptqmodel"),
    "turboquant": get_pkg_version("turboquant"),
    "pynvml": get_pkg_version("pynvml"),
    "numpy": get_pkg_version("numpy"),
    "scipy": get_pkg_version("scipy"),
    "pandas": get_pkg_version("pandas"),
    "matplotlib": get_pkg_version("matplotlib"),
}

# Reference pinned stack from registry for comparison
canonical_software_pins = {
    "torch": "2.11.0+cu128",
    "cuda_runtime": "12.8",
    "transformers": "5.12.1",
    "transformers_commit": "ddb849abe009d1089e6c691bfc897f27211c663c",
    "turboquant": "0.2.0",
    "turboquant_source_sha256": "7358e2778c8de6974b0c7212ae8e3dbe00552a96b2e2df0aa42878f239a4c72a",
    "gptqmodel": "7.1.0",
    "accelerate": "1.14.0",
    "numpy": "2.2.6",
    "scipy": "1.16.3 (Llama) / 1.16.1 (Qwen)",
    "pandas": "2.2.2",
}

print("Active Software Stack:")
print(json.dumps(software_stack, indent=2))
print("\nCanonical Pinned Revisions:")
print(json.dumps(canonical_software_pins, indent=2))


## 5. Model Architecture and Checkpoint Geometry

In [ ]:
models_spec = {
    "llama_3_2_3b_instruct": {
        "model_family": "Llama 3.2",
        "parameter_count": "3.21B",
        "uncompressed_repo_id": "unsloth/Llama-3.2-3B-Instruct",
        "uncompressed_revision": "006f5dcd1393c3add266de40994ba96225e9689d",
        "awq_repo_id": "casperhansen/llama-3.2-3b-instruct-awq",
        "awq_revision": "272b3bde867b606760447deb9a4d2719fbdfd3ae",
        "awq_quantization_config": {
            "bits": 4,
            "group_size": 128,
            "zero_point": True,
            "version": "GEMM",
        },
        "architecture": {
            "num_hidden_layers": 28,
            "num_attention_heads_query": 24,
            "num_key_value_heads": 8,
            "gqa_ratio": "3:1 (24 Q / 8 KV)",
            "head_dim": 128,
            "hidden_size": 3072,
            "intermediate_size": 8192,
            "vocab_size": 128256,
            "max_position_embeddings": 8192,
            "rope_theta": 500000.0,
            "hidden_act": "silu",
        },
        "kv_cache_bytes_per_token_fp16": 28 * 2 * 8 * 128 * 2, # 114,688 bytes = 112 KiB / token
        "kv_cache_mib_at_4096_fp16": round((4096 * 28 * 2 * 8 * 128 * 2) / (1024**2), 2), # 448.00 MiB (raw) / 455 MiB live allocated
    },
    "qwen_2_5_3b_instruct": {
        "model_family": "Qwen 2.5",
        "parameter_count": "3.09B",
        "uncompressed_repo_id": "Qwen/Qwen2.5-3B-Instruct",
        "uncompressed_revision": "aa8e72537993ba99e69dfaafa59ed015b17504d1",
        "awq_repo_id": "Qwen/Qwen2.5-3B-Instruct-AWQ",
        "awq_revision": "3559b226e8ce77211e2c1bd7ddfb7686fec4d6dd",
        "awq_quantization_config": {
            "bits": 4,
            "group_size": 128,
            "zero_point": True,
            "version": "GEMM",
        },
        "architecture": {
            "num_hidden_layers": 36,
            "num_attention_heads_query": 16,
            "num_key_value_heads": 2,
            "gqa_ratio": "8:1 (16 Q / 2 KV)",
            "head_dim": 128,
            "hidden_size": 2048,
            "intermediate_size": 11008,
            "vocab_size": 151936,
            "max_position_embeddings": 32768,
            "rope_theta": 1000000.0,
            "hidden_act": "silu",
        },
        "kv_cache_bytes_per_token_fp16": 36 * 2 * 2 * 128 * 2, # 36,864 bytes = 36 KiB / token
        "kv_cache_mib_at_4096_fp16": round((4096 * 36 * 2 * 2 * 128 * 2) / (1024**2), 2), # 144.00 MiB (raw) / 146.25 MiB live allocated
    }
}

print(json.dumps(models_spec, indent=2))


## 6. Compression Protocols and Layerwise Eviction Schedules

In [ ]:
def calculate_pyramid_budgets(num_layers, avg_budget=1024, beta=20, min_ratio=0.08):
    """Calculates exact layerwise PyramidKV retention budgets."""
    # Linear decay baseline before beta scaling
    raw_weights = [((num_layers - 1 - l) / (num_layers - 1)) ** (1.0 / beta) for l in range(num_layers)]
    # Normalize to match total target budget
    scale = (avg_budget * num_layers) / sum(raw_weights)
    budgets = [int(round(w * scale)) for w in raw_weights]
    return budgets

llama_budgets = calculate_pyramid_budgets(28, avg_budget=1024)
qwen_budgets = calculate_pyramid_budgets(36, avg_budget=1024)

compression_protocols = {
    "weight_quantization": {
        "method": "Activation-aware Weight Quantization (AWQ)",
        "precision": "INT4 GEMM",
        "group_size": 128,
        "weight_storage_llama_gib": 1.76,
        "weight_storage_qwen_gib": 1.71,
    },
    "token_eviction": {
        "policy": "Custom Pyramidal KV-cache Eviction (PyramidKV-style)",
        "average_budget_tokens": 1024,
        "sensitivity_budgets_tokens": [512, 1024, 1536],
        "attention_window": 32,
        "pooling_kernel_size": 5,
        "llama_28layer_allocation": llama_budgets,
        "qwen_36layer_allocation_range": {
            "layer_0_max": 1966,
            "layer_35_min": 82,
            "average": 1024,
        },
    },
    "kv_quantization": {
        "framework": "TurboQuant reference implementation (turboquant==0.2.0)",
        "nominal_bit_widths": [2, 3, 4],
        "physical_storage_format": "uint8 index containers + FP32 norms + uncompressed FP16 residual",
        "uncompressed_residual_window_tokens": 128,
        "reconstruction_error_bound_nmse": {
            "2_bit": "<= 1.00",
            "3_bit": "<= 0.50",
            "4_bit": "<= 0.25 (observed ~2.5e-4 on 4-bit)",
        },
    },
    "sequential_pipeline_execution": {
        "stage_1": "AWQ linear projection inference (INT4 weights, FP16 activations)",
        "stage_2": "Attention scoring and pyramidal token eviction pruning",
        "stage_3": "TurboQuant coordinate quantization on retained tokens (TQ-Ref)",
    }
}

print(json.dumps(compression_protocols, indent=2))


## 7. Experimental Evaluation Protocol and Statistical Settings

In [ ]:
evaluation_protocol = {
    "evaluation_task": "Hard-Retrieval Needle-In-A-Haystack (NIAH) Key-Value Extraction",
    "test_scenarios_total": 60,
    "depth_distribution": {
        "shallow_10_percent": 20,
        "middle_50_percent": 20,
        "deep_90_percent": 20,
    },
    "sensitivity_scenarios_common": 15,
    "context_lengths_evaluated": [1024, 2048, 4096],
    "fixed_decode_length_tokens": 64,
    "generation_sampling": {
        "do_sample": False,
        "temperature": 0.0,
        "top_p": 1.0,
    },
    "random_seed": 407,
    "energy_measurement": {
        "blocks_per_condition": 5,
        "sampling_frequency_hz": 10.0,
        "integration": "Trapezoidal rule",
        "metrics": ["Total Energy (J)", "Prefill Energy (J)", "Decode Energy (J)", "Decode Throughput (tok/s)", "Peak Allocated VRAM (MiB)"],
    },
    "statistical_methods": {
        "energy_pairwise_tests": "Paired Student's t-test and Wilcoxon signed-rank test",
        "multiple_comparisons_correction": "Holm-Bonferroni step-down procedure (family alpha = 0.05)",
        "quality_pairwise_tests": "Exact paired McNemar test with Bonferroni correction",
        "quality_confidence_intervals": "Wilson score 95% confidence intervals",
        "pareto_frontier_uncertainty": "2000 depth-stratified bootstrap resamples",
        "sequential_interaction_model": "I = E(C4) - E(C1) - [E(C2) - E(C1)] - [E(C3) - E(C1)]",
    }
}

print(json.dumps(evaluation_protocol, indent=2))


## 8. Export Reproducibility Artifacts (JSON, LaTeX, Markdown)

In [ ]:
# 1. Compile master dictionary
master_specs = {
    "schema_version": "cse407-reproducibility-audit-v1",
    "audit_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "host_specs": host_specs,
    "gpu_hardware": gpu_specs,
    "software_stack": software_stack,
    "canonical_software_pins": canonical_software_pins,
    "model_architectures": models_spec,
    "compression_protocols": compression_protocols,
    "evaluation_protocol": evaluation_protocol,
}

output_dir = Path("./reproducibility_export")
output_dir.mkdir(exist_ok=True)

# Save full JSON
json_path = output_dir / "reproducibility_specs_full.json"
json_path.write_text(json.dumps(master_specs, indent=2))
print(f"Saved JSON: {json_path} ({json_path.stat().st_size} bytes)")

# 2. Generate publication LaTeX table
latex_table = r"""\begin{table}[htbp]
\caption{Detailed Reproducibility Specifications and Environmental Parameters}
\label{tab:reproducibility-specs}
\centering\small
\resizebox{\columnwidth}{!}{%
\begin{tabular}{@{}ll@{}}
\toprule
\textbf{Parameter Category} & \textbf{Configuration / Specification} \\
\midrule
\multicolumn{2}{l}{\textit{Hardware and Telemetry}} \\
GPU Architecture & NVIDIA Tesla T4 (Turing, Compute Capability 7.5) \\
VRAM Capacity & 16~GB nominal (14.56~GiB addressable, 256-bit bus) \\
Enforced Power Limit & 70.0~W default and enforced cap (NVML sampled) \\
Power Sampling & 10~Hz (100~ms interval) via \texttt{pynvml} \\
Power Integration & Trapezoidal Riemann summation $\int P(t)\,dt$ \\
Robustness Caps & Raw trace, 82~W clipped, 70~W clipped \\
\midrule
\multicolumn{2}{l}{\textit{Software Runtime and Libraries}} \\
Operating Environment & Ubuntu 22.04 LTS on Google Colab \\
Deep Learning Framework & PyTorch 2.11.0+cu128, CUDA 12.8 \\
Model Implementation & Hugging Face Transformers 5.12.1 (pinned commit) \\
Quantization Libraries & \texttt{gptqmodel==7.1.0}, \texttt{accelerate==1.14.0} \\
KV Quantization & \texttt{turboquant==0.2.0} (pinned source SHA-256) \\
Scientific Stack & Python 3.11, NumPy 2.2.6, SciPy 1.16.1/3, Pandas 2.2.2 \\
\midrule
\multicolumn{2}{l}{\textit{Model Checkpoints and Geometry}} \\
Llama 3.2-3B-Instruct & 28 layers, 24 Q / 8 KV heads (3:1 GQA), head dim 128 \\
Llama FP16 / AWQ Rev. & \texttt{006f5dcd...} / \texttt{272b3bde...} \\
Qwen2.5-3B-Instruct & 36 layers, 16 Q / 2 KV heads (8:1 GQA), head dim 128 \\
Qwen FP16 / AWQ Rev. & \texttt{aa8e7253...} / \texttt{3559b226...} \\
\midrule
\multicolumn{2}{l}{\textit{Compression and Eviction Configuration}} \\
Weight Quantization & AWQ GEMM INT4, group size 128, zero-point true \\
Token Eviction Policy & Custom PyramidKV (decay $\beta=20$, win 32, kernel 5) \\
Eviction Budget & Avg 1024 tokens (Llama: 28 layers; Qwen: 1966 to 82) \\
TurboQuant Reference & Coordinate uint8 indices, FP32 norms, 128-tok FP16 res. \\
Sequential Stacking & AWQ linear $\to$ attention eviction $\to$ TQ reference \\
\midrule
\multicolumn{2}{l}{\textit{Evaluation and Statistical Protocols}} \\
Quality Benchmark & Hard-retrieval NIAH UUID extraction (60 scenarios) \\
Context Lengths & 1024, 2048, 4096 tokens (64-token fixed decode) \\
Energy Sample Blocks & $n=5$ independent measurement runs per condition \\
Statistical Inference & Paired $t$-test / Wilcoxon with Holm correction \\
Categorical Tests & Exact McNemar test with Bonferroni adjustment \\
Pareto Uncertainty & 2000 depth-stratified bootstrap resamples \\
\bottomrule
\end{tabular}}
\end{table}
"""

tex_path = output_dir / "reproducibility_table.tex"
tex_path.write_text(latex_table)
print(f"Saved LaTeX Table: {tex_path} ({tex_path.stat().st_size} bytes)")

# 3. Generate Markdown summary
md_summary = f"""# Reproducibility Specification Reference Sheet

**Generated:** {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}  
**Schema:** `cse407-reproducibility-audit-v1`

## 1. Hardware & Power Sampling
- **GPU:** NVIDIA Tesla T4 (Turing, Compute Capability 7.5, 40 SMs)
- **VRAM:** 16 GB nominal / 14.56 GiB usable (256-bit bus, ~320 GB/s bandwidth)
- **Power Limit:** 70.0 W enforced cap
- **Power Telemetry:** 10 Hz NVML sampling (100 ms interval), trapezoidal Riemann integration
- **Power Robustness Verification:** Reintegrated under raw, 82 W cap, and 70 W cap

## 2. Software Stack & Pinned Revisions
- **CUDA / Driver:** CUDA 12.8 / Driver 550.54.15
- **PyTorch:** 2.11.0+cu128
- **Transformers:** 5.12.1 (`ddb849abe009d1089e6c691bfc897f27211c663c`)
- **TurboQuant:** 0.2.0 (SHA-256: `7358e2778c8de6974b0c7212ae8e3dbe00552a96b2e2df0aa42878f239a4c72a`)
- **GPTQModel:** 7.1.0
- **Accelerate:** 1.14.0
- **Python:** 3.11 (NumPy 2.2.6, SciPy 1.16.1/3, Pandas 2.2.2)

## 3. Evaluated Model Geometry
- **Llama 3.2-3B-Instruct:**
  - FP16: `unsloth/Llama-3.2-3B-Instruct` (rev: `006f5dcd1393c3add266de40994ba96225e9689d`)
  - AWQ: `casperhansen/llama-3.2-3b-instruct-awq` (rev: `272b3bde867b606760447deb9a4d2719fbdfd3ae`)
  - Geometry: 28 layers, 24 Q / 8 KV heads (3:1 GQA), head dim 128, 8K context
- **Qwen2.5-3B-Instruct:**
  - FP16: `Qwen/Qwen2.5-3B-Instruct` (rev: `aa8e72537993ba99e69dfaafa59ed015b17504d1`)
  - AWQ: `Qwen/Qwen2.5-3B-Instruct-AWQ` (rev: `3559b226e8ce77211e2c1bd7ddfb7686fec4d6dd`)
  - Geometry: 36 layers, 16 Q / 2 KV heads (8:1 GQA), head dim 128, 32K context

## 4. Compression Pipelines & Parameters
- **Weight Quantization:** AWQ GEMM INT4, group size 128
- **Token Eviction:** Custom PyramidKV (geometric decay beta=20, pool kernel=5, window=32, avg budget=1024 tokens)
- **KV Quantization:** TurboQuant 0.2.0 numerical reference (uint8 index container, FP32 norms, 128-token FP16 residual)
- **Sequential Compound:** AWQ -> PyramidKV eviction -> TurboQuant reference

## 5. Experimental Protocols & Statistical Tests
- **Quality Evaluation:** Hard-retrieval Needle-in-a-Haystack (NIAH) 60 scenarios (20 at 10%, 20 at 50%, 20 at 90% depth)
- **Decoding:** Teacher-forced 64-token decode, greedy generation (temp=0.0)
- **Energy Samples:** n=5 blocks per condition
- **Statistical Corrections:** Paired t-tests / Wilcoxon with Holm-Bonferroni correction (alpha=0.05); exact McNemar with Bonferroni correction; Wilson 95% CIs; 2000 bootstrap resamples for Pareto stability
"""

md_path = output_dir / "reproducibility_summary.md"
md_path.write_text(md_summary)
print(f"Saved Markdown Summary: {md_path} ({md_path.stat().st_size} bytes)")


## 9. Verification and Download Utility

In [ ]:
import shutil

zip_path = Path("./reproducibility_bundle.zip")
shutil.make_archive("reproducibility_bundle", "zip", "./reproducibility_export")
print(f"Created archive: {zip_path} ({zip_path.stat().st_size} bytes)")

# If running in Google Colab, trigger direct browser download
try:
    from google.colab import files
    files.download(str(zip_path))
    print("Triggered Colab file download for reproducibility_bundle.zip")
except ImportError:
    print("Local execution mode: files are available in ./reproducibility_export/")
